# Context Relevance Evaluator

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://www.shutterstock.com/image-vector/concept-context-magnifying-glass-letters-600nw-2482176791.jpg"> 
</p>
</div>

## Description:

This app analyzes AI-generated responses to determine their relevance to a given context. It assigns a score, provides insights, and ensures meaningful interactions.

- Evaluates response relevance based on provided context. 

- Uses OpenAI’s models for scoring and reflection.  

- Helps maintain accuracy and coherence in AI responses.  

- Provides detailed feedback on context adherence.  

- Includes robust error handling for reliability.  



## Step 1: Environment Setup and Installation

This step installs dependencies from `requirements.txt` and checks for `OPENAI_API_KEY`.  

If installation fails, it retries up to 3 times before exiting.  

Once complete, it clears the output and prints a success message.  


In [ ]:
# Boilerplate: This block goes into every notebook.
# It sets up the environment, installs the requirements, and checks for the required environment variables.

from IPython.display import clear_output
import os

requirements_installed = False
max_retries = 3
retries = 0
REQUIRED_ENV_VARS = ["OPENAI_API_KEY"]


def install_requirements():
    """Installs the requirements from requirements.txt file"""
    global requirements_installed, retries
    if requirements_installed:
        print("Requirements already installed.")
        return

    print("Installing requirements...")
    install_status = os.system("pip install -r requirements.txt")
    if install_status == 0:
        print("Requirements installed successfully.")
        requirements_installed = True
    else:
        print("Failed to install requirements.")
        if retries < max_retries:
            print("Retrying...")
            retries += 1
            return install_requirements()
        exit(1)
    return


install_requirements()
clear_output()
print("🚀 Setup complete. Continue to the next cell.")

## Step 2: Environment Variable Setup

This step loads environment variables from `.env` using `dotenv`.  

It checks if `OPENAI_API_KEY` is set; if missing, it exits.  

After validation, it confirms successful setup.  


In [ ]:
from dotenv import load_dotenv

def setup_env():
    """Sets up the environment variables"""

    def check_env(env_var):
        value = os.getenv(env_var)
        if value is None:
            print(f"Please set the {env_var} environment variable.")
            exit(1)
        else:
            print(f"{env_var} is set.")

    load_dotenv(override=True, dotenv_path=".env")

    variables_to_check = REQUIRED_ENV_VARS

    for var in variables_to_check:
        check_env(var)

    print("Environment variables are set.")


setup_env()

In [ ]:
import json
from composio_openai import ComposioToolSet, Action
from openai import OpenAI
import traceback

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") # Replace with your OpenAI API key
COMPOSIO_API_KEY = os.getenv("COMPOSIO_API_KEY") # Replace with your Composio API key
AUTHOR_ID = os.getenv("AUTHOR_ID") # Replace with your LinkedIn Author ID, To find your author ID, check out the request_test.py file in the scripts folder 

if not OPENAI_API_KEY or not COMPOSIO_API_KEY or not AUTHOR_ID:
    raise ValueError("Missing API keys or AUTHOR_ID. Ensure you have set the OPENAI_API_KEY and COMPOSIO_API_KEY in your .env file.")


openai_client = OpenAI(api_key=OPENAI_API_KEY) 
composio_toolset = ComposioToolSet(api_key=COMPOSIO_API_KEY)

post_content = {
    "post": {
        "author": AUTHOR_ID, # Replace with your LinkedIn Author ID
        "commentary": (
           #Insert your LinkedIn post content here. Be sure to include the key insights and a strong call to action.
        ),
        "visibility": "PUBLIC"
    }
}


def create_linkedin_post(post_content: dict) -> dict | None:
    """
    Creates a LinkedIn post using Composio's LinkedIn tool.

    Args:
        post_content (dict): A dictionary containing the post content.
    
    Returns:
        dict: The response from the LinkedIn tool.
    """

    if not post_content:
        raise ValueError("❌ Post content cannot be empty!")
    
    try :
        print("Creating LinkedIn post...")

        tools = composio_toolset.get_tools(actions=[Action.LINKEDIN_CREATE_LINKED_IN_POST]) 

        SYSTEM_PROMPT = """
        You are an expert in writing engaging and professional LinkedIn posts. 
        Craft a compelling LinkedIn post using the provided details while ensuring:
        ✅ Clarity, impact, and readability.
        ✅ A structured format with an engaging introduction, key insights, and a strong call to action.
        ✅ A professional yet conversational tone to maximize audience engagement.
        """

        response = openai_client.chat.completions.create(
        model="gpt-4o",
        tools=tools,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},   
            {"role": "user", "content": f"POST_CONTENT: {json.dumps(post_content, indent=2)}"},
        ],
    )
        result = composio_toolset.handle_tool_calls(response)
        if not result:
            print("❌ API returned an empty response!")   

        print("✅ LinkedIn post created successfully!")
        return result
    
    except Exception as e:
        print(f"❌ Error creating LinkedIn post: {str(e)}")
        traceback.print_exc()
        return None
        
if __name__ == "__main__":
    response = create_linkedin_post(post_content)
    if response:
        print("✅ LinkedIn post created successfully!")
    else:
        print("🚨 FAILED TO CREATE LINKEDIN POST")